# Notebook 04 — Analysis and Plots

**Member responsible: C**

Loads all results CSVs and generates plots for the report.

Plots produced:
1. Accuracy comparison across all experiments (bar chart)
2. Accuracy vs latency trade-off (scatter plot)
3. A2 ablation: accuracy vs LoRA rank (line chart)
4. A3 ablation: accuracy vs training data size (learning curve)
5. D2 vs D1: effect of filtering on accuracy (bar chart)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
RESULTS_DIR = Path('results/')
PLOT_DIR = Path('results/plots/')
PLOT_DIR.mkdir(exist_ok=True)

In [ ]:
# -------------------------------------------------------------------------
# Load all experiment results
# TODO: update filenames to match what was actually saved in notebook 03
# -------------------------------------------------------------------------
from src.evaluation.metrics import compute_summary_stats

def load_and_summarise(csv_path: str, exp_name: str) -> dict:
    df = pd.read_csv(csv_path)
    results = df.to_dict(orient='records')
    stats = compute_summary_stats(results)
    stats['experiment'] = exp_name
    return stats

# TODO: add/remove rows as experiments complete
experiment_files = [
    ('exp0_zero_shot_gsm8k.csv',          'Exp 0: Zero-shot'),
    ('exp1_few_shot_gsm8k.csv',           'Exp 1: Few-shot CoT'),
    ('exp2_adaptation_r16_gsm8k.csv',     'Exp 2: Adaptation (LoRA r=16)'),
    ('exp3_distillation_gsm8k.csv',       'Exp 3: Distillation (D1)'),
    ('exp4_filtered_gsm8k.csv',           'Exp 4: Filtered Distill. (D2)'),
    ('exp5_d3_cross_task_arc.csv',        'Exp 5: Cross-task (D3, ARC)'),
]

summaries = []
for fname, exp_name in experiment_files:
    fpath = RESULTS_DIR / fname
    if fpath.exists():
        summaries.append(load_and_summarise(str(fpath), exp_name))
    else:
        print(f'[SKIP] {fname} not found yet')

summary_df = pd.DataFrame(summaries)
summary_df

In [ ]:
# -------------------------------------------------------------------------
# Plot 1: Accuracy comparison across all experiments
# -------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=summary_df, x='experiment', y='accuracy', ax=ax)
ax.set_title('Accuracy by Experiment (GSM8K)', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot1_accuracy_comparison.png', dpi=150)
plt.show()

In [ ]:
# -------------------------------------------------------------------------
# Plot 2: Accuracy vs latency trade-off
# Key figure for the report — shows the cost/accuracy frontier
# -------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=summary_df,
    x='mean_latency_ms',
    y='accuracy',
    hue='experiment',
    s=120,
    ax=ax
)
for _, row in summary_df.iterrows():
    ax.annotate(row['experiment'], (row['mean_latency_ms'], row['accuracy']),
                textcoords='offset points', xytext=(5, 5), fontsize=8)
ax.set_title('Accuracy vs Latency Trade-off', fontsize=13)
ax.set_xlabel('Mean Latency (ms/sample)')
ax.set_ylabel('Accuracy')
ax.legend().remove()
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot2_accuracy_vs_latency.png', dpi=150)
plt.show()

In [ ]:
# -------------------------------------------------------------------------
# Plot 3: A2 ablation — accuracy vs LoRA rank
# TODO: fill in actual accuracy values after running A2 ablation
# -------------------------------------------------------------------------
a2_data = pd.DataFrame({
    'rank': [4, 8, 16, 32],
    'accuracy': [None, None, None, None],  # TODO: fill in after runs
    'mean_latency_ms': [None, None, None, None],
})

fig, ax1 = plt.subplots(figsize=(7, 4))
ax2 = ax1.twinx()
ax1.plot(a2_data['rank'], a2_data['accuracy'], marker='o', label='Accuracy', color='steelblue')
ax2.plot(a2_data['rank'], a2_data['mean_latency_ms'], marker='s', label='Latency', color='coral', linestyle='--')
ax1.set_xlabel('LoRA Rank')
ax1.set_ylabel('Accuracy', color='steelblue')
ax2.set_ylabel('Mean Latency (ms)', color='coral')
ax1.set_title('A2 Ablation: Accuracy vs LoRA Rank')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot3_a2_rank_ablation.png', dpi=150)
plt.show()

In [ ]:
# -------------------------------------------------------------------------
# Plot 4: A3 ablation — learning curve (accuracy vs training data size)
# TODO: fill in actual accuracy values after running A3 ablation
# -------------------------------------------------------------------------
a3_data = pd.DataFrame({
    'fraction': [0.1, 0.25, 0.5, 1.0],
    'n_samples': [747, 1868, 3736, 7473],
    'accuracy': [None, None, None, None],  # TODO: fill in after runs
})

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(a3_data['n_samples'], a3_data['accuracy'], marker='o', color='mediumseagreen')
ax.set_xlabel('Training Samples')
ax.set_ylabel('Accuracy')
ax.set_title('A3 Ablation: Learning Curve (Accuracy vs Training Size)')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot4_a3_data_size_ablation.png', dpi=150)
plt.show()

In [ ]:
# -------------------------------------------------------------------------
# Save summary table as CSV for the report
# -------------------------------------------------------------------------
summary_df.to_csv(RESULTS_DIR / 'summary_all_experiments.csv', index=False)
print('Summary saved.')
summary_df[['experiment', 'accuracy', 'mean_latency_ms', 'mean_token_count', 'n_samples']]